In [1]:
import torch
import mlflow
import evaluate
import numpy as np 
from datasets import load_dataset
from transformers import (
    TrainingArguments, 
    Trainer, 
    DistilBertForSequenceClassification, 
    DistilBertTokenizer, 
    DataCollatorWithPadding, 
    EvalPrediction
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Train Model

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Distilbert Fine Tuning")
mlflow.autolog()   

2026/05/15 01:20:35 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/05/15 01:20:35 INFO mlflow.tracking.fluent: Autologging successfully enabled for transformers.


In [5]:
if torch.cuda.is_available():
    print("Running training with on CUDA!")

Running training with on CUDA!


In [6]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [7]:
def tokenize(batch):
    tokens =  tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
    )

    tokens['labels'] = batch['label']

    return tokens

dataset = load_dataset("imdb", cache_dir="data/cache", split="train")
dataset = dataset.map(tokenize, batched=True)
dataset = dataset.train_test_split(test_size=0.2)

In [8]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
args = TrainingArguments(
    output_dir="models/distilbert",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    dataloader_pin_memory=False,
    seed=42,
    logging_dir="logs/"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred: EvalPrediction):
    predictions, labels = eval_pred
    
    # Convert logits to predicted class indices
    predictions = np.argmax(predictions, axis=1)
    
    # Calculate individual metrics
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average='weighted')
    precision = precision_metric.compute(predictions=predictions, references=labels, average='weighted')
    recall = recall_metric.compute(predictions=predictions, references=labels, average='weighted')
    
    return {
        'accuracy': accuracy['accuracy'],
        'f1': f1['f1'],
        'precision': precision['precision'],
        'recall': recall['recall']
    }

In [11]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=dataset["train"], 
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.248743,0.198162,0.921600,0.921569,0.921841,0.921600
2,0.168075,0.231980,0.927200,0.927206,0.927330,0.927200
3,0.096076,0.304284,0.925600,0.925606,0.925718,0.925600
4,0.053205,0.396680,0.920800,0.920804,0.921149,0.920800
5,0.021522,0.383851,0.927400,0.927393,0.927424,0.927400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6250, training_loss=0.12097398361206055, metrics={'train_runtime': 985.455, 'train_samples_per_second': 101.476, 'train_steps_per_second': 6.342, 'total_flos': 1.3115501929987776e+16, 'train_loss': 0.12097398361206055, 'epoch': 5.0})

In [13]:
trainer.save_model()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

trainer.evaluate()